##流式输出

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from openai import AsyncOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)
for chunk in model.stream("写一首七言律诗，总结大模型的发展"):
    print(chunk.text, end="", flush=True)  # 逐token输出

##批量调用:batch()方法

In [ ]:
message = [
    "你好吗今天",
    "1+2+3=?",
    "今天好热适合干什么"
]
# response = model.batch(message)   #整个的输出
streamType = model.stream(message)  #流式

for chunk in streamType :   #流式输出模式
    print(chunk.text, end="", flush=True)


##按照顺序输出, 三个问题，(0,第一个问题),(1,第二个).(2,第三个)

In [ ]:
message = [
    "你好吗今天",
    "1+12=?",
    "今天好热适合干什么"
]
streamType = model.batch_as_completed(message)  #流式

for chunk in streamType :   #流式输出模式
    print(chunk, end="", flush=True)

##异步操做

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
#
client = AsyncOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

#=================
#   单个异步
#=================
async def single():
    response = await client.chat.completions.create(
        model = "deepseek-chat",
        messages=[{"role": "user","content":"介绍你自己"}],
        stream=True
    )
    #print(response)

#异步循环
    async for chunk in response:
        content = chunk.choices[0].delta.content
        if content is not None:
            print(content, end="", flush=True)
print()
# ============================================
# 2. 并发多个请求
# ============================================
async def concurrent():
    questions = ["什么是skill？", "什么是RAG？", "1+1等于几？"]

    tasks = [
        client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": q}]
        )
        for q in questions
    ]

    results = await asyncio.gather(*tasks)

    for i, r in enumerate(results):
        print(f"{questions[i]}: {r.choices[0].message.content[:50]}...")

    print("\n--- 并发请求结束 ---\n")
await concurrent()
await single()


##异步流

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv(override=True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# ============================================
# 1. 基础异步流式输出（实时打字效果）
# ============================================
async def basic_stream():
    stream = await client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "你喜欢哪两个字"}],
        stream=True
    )

    print("🤖: ", end="")
    async for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print("\n")

# ============================================
# 2. 并发多个流式请求
# ============================================
async def concurrent_stream():
    questions = [
        "什么是Python？",
        "什么是异步？",
        "解释一下API"
    ]

    async def stream_question(q):
        print(f"\n📝 问题: {q}")
        print("🤖: ", end="")

        stream = await client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": q}],
            stream=True
        )

        full_response = ""
        async for chunk in stream:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        print("\n")
        return full_response

    # 并发执行所有流式请求
    tasks = [stream_question(q) for q in questions]
    await asyncio.gather(*tasks)

# ============================================
# 3. 流式 + 超时控制
# ============================================
async def stream_with_timeout():
    try:
        stream = await client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": "详细解释量子计算，要很详细"}],
            stream=True
        )

        print("🤖: ", end="")
        async for chunk in stream:
            if chunk.choices[0].delta.content:
                print(chunk.choices[0].delta.content, end="", flush=True)
        print("\n✅ 流式完成")

    except asyncio.TimeoutError:
        print("\n⏰ 请求超时")
    except Exception as e:
        print(f"\n❌ 错误: {e}")

# ============================================
# 主函数
# ============================================
async def main():
    print("=" * 50)
    print("1. 基础流式输出")
    print("=" * 50)
    await basic_stream()

    print("=" * 50)
    print("2. 并发流式输出")
    print("=" * 50)
    await concurrent_stream()

    print("=" * 50)
    print("3. 流式 + 超时")
    print("=" * 50)
    await stream_with_timeout()

if __name__ == "__main__":
    await main()